# Steerling-8B Instruct: Text Generation

Steerling Instruct is a **causal diffusion language model** fine-tuned for instruction following.
It uses the same block-based denoising generation as the base model, but with a chat template
that supports multi-turn conversations with system, user, and assistant roles.

The chat format follows a Llama 3-style template:
```
<|start_header_id|>system<|end_header_id|>\n\n{system_message}<|eot_id|>
<|start_header_id|>user<|end_header_id|>\n\n{user_message}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>\n\n{assistant_response}<|eot_id|>
```

**Requirements:** GPU with >= 18 GB VRAM (A100, A6000, RTX 4090)

## 1. Load Model

We load the instruct model via HuggingFace `AutoModel` and wrap it with `SteerlingGenerator`.

First run downloads ~17 GB from HuggingFace Hub; subsequent runs load from cache.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig

model_id = "guidelabs/steerling-8b-instruct"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")
print(generator)
print(f"\nBlock size: {generator.diff_block_size} tokens")
print(f"Interpretable: {generator.is_interpretable}")

## 2. Chat Template

The instruct model uses a chat template to format conversations. The tokenizer's
`apply_chat_template` method converts a list of messages into the format the model expects.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "What is a diffusion language model?"}
]

# See what the chat template produces
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("Formatted prompt:")
print(repr(formatted))

## 3. Generate with Chat Template

We apply the chat template to format the prompt, then pass the formatted string to the generator.

Key parameters:
- **`max_new_tokens`** — how many tokens to generate. Must be divisible by `block_length` (default 64).
- **`steps`** — total denoising steps, split evenly across blocks. More steps = more refinement. Defaults to `max_new_tokens` if not set.
- **`temperature`** — base temperature for sampling (used with `top_p`; ignored when `use_entropy_sampling=True`).
- **`top_p`** — nucleus sampling threshold. Keeps tokens whose cumulative probability stays within this.
- **`use_entropy_sampling`** — adaptive temperature based on per-token entropy (recommended for instruct).
- **`repetition_penalty`** — penalizes repeated tokens (>1.0 reduces repetition).
- **`stop_tokens`** — token IDs that trigger early stopping. Use `<|eot_id|>` for instruct.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "What is a diffusion language model?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Uses production defaults: entropy sampling, top_p=0.8, rep_penalty=1.1
config = GenerationConfig(
    steps=1024,
    seed=42,
    stop_tokens=[tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

text = generator.generate(prompt, config)
print(text)

## 4. Multi-Turn Conversation

The chat template supports multi-turn conversations. Include previous assistant responses
in the message history to maintain context.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "What is a diffusion language model?"},
    {"role": "assistant", "content": "A diffusion language model generates text by iteratively denoising masked tokens rather than predicting one token at a time left-to-right."},
    {"role": "user", "content": "How is that different from GPT?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

config = GenerationConfig(
    steps=1024,
    seed=42,
    stop_tokens=[tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

text = generator.generate(prompt, config)
print(text)

## 5. Different Prompts

A few examples showing the instruct model on different tasks.

In [ ]:
examples = [
    "Explain quantum computing in simple terms.",
    "Write a Python function that checks if a string is a palindrome.",
    "What are the main differences between Python and Rust?",
]

config = GenerationConfig(
    steps=1024,
    seed=42,
    stop_tokens=[tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

for user_msg in examples:
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
        {"role": "user", "content": user_msg}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text = generator.generate(prompt, config)

    print(f"{'='*60}")
    print(f"User: {user_msg}")
    print(f"Assistant:")
    print(text)
    print()

## Generation Parameters Reference

| Parameter              | Default          | Description                                                        |
|------------------------|------------------|--------------------------------------------------------------------|
| `max_new_tokens`       | 1024             | Number of tokens to generate                                       |
| `steps`                | `max_new_tokens` | Total denoising steps (split evenly across blocks)                 |
| `temperature`          | 1.2              | Sampling temperature (ignored when `use_entropy_sampling=True`)    |
| `top_p`                | 0.8              | Nucleus sampling threshold                                         |
| `repetition_penalty`   | 1.1              | Penalizes repeated tokens (1.0 = disabled)                         |
| `use_entropy_sampling` | True             | Adaptive temperature based on per-token entropy                    |
| `cfg_scale`            | 0.0              | Classifier-free guidance scale (0 = disabled)                      |
| `seed`                 | None             | Random seed for reproducibility                                    |
| `stop_tokens`          | None             | Token IDs that trigger early stop between blocks                   |

**Tip:** For the instruct model, always use `stop_tokens=[tokenizer.convert_tokens_to_ids("<|eot_id|>")]` to stop at the end of the assistant's response.

**Tip:** Reducing `steps` unmasks multiple tokens per step — faster generation but lower quality.

**Constraints:**
- `max_new_tokens` must be divisible by `block_length` (default 64)
- `steps` must be divisible by the number of blocks (`max_new_tokens / block_length`)